# Real-robot peg-insert evaluation — method comparison

Analysis of the real FR3 FORGE peg-insert eval (`data/real_robot_eval/<method>/<agent>/`).
Three **methods** (`contact_baseline`, `contact_hist8`, `dyn_pinv`), each with **5 agents**,
each agent run for **20 trajectories** (100 trajectories per method).

Per-episode outcome + force metrics are reconstructed from the per-step episode CSVs by
replaying the eval's own terminal logic (`peg_insert_eval.run_episode`): a trajectory
**BREAKs** when `force_mag >= break_force` (10 N) and **SUCCEEDs** when
`xy_dist_to_target < xy_centering_threshold` **and** `z_disp < hole_height*success_threshold`
(`terminate_on_success`). This replay reproduces the one ground-truth `summary.csv`
(`contact_baseline/0`) exactly — every outcome and every force metric.

**Per-trajectory metrics**
- **success** — reached the seated-peg tolerance before any break/timeout.
- **break** — hit the force limit (`force_mag >= break_force`).
- **average force** — mean `force_mag` over the logical trajectory (`sum_force / data_length`, N).
- **max force** — peak `force_mag` over the logical trajectory (N).

**Plots**
1. *Among agents* (n = 5): 2x2 bars (success, break, avg force, max force), 95% CI across the 5 agents.
2. *Per trajectory* (n = 100): the same 2x2, treating every trajectory as one sample.
3. Stacked success/break **counts** — one stack per method, one segment per agent.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

## Global variables

In [ ]:
# ---- paths (relative to this notebook in data_analysis/) ----
DATA_DIR = "../data/real_robot_eval"
FIG_DIR  = "real_robot_eval_figs"
os.makedirs(FIG_DIR, exist_ok=True)

# ---- experiment layout ----
METHODS = ["contact_baseline", "contact_hist8", "dyn_pinv"]   # plotting order
METHOD_LABELS = {"contact_baseline": "Contact\nBaseline",
                 "contact_hist8":    "Contact\nHist-8",
                 "dyn_pinv":         "Dyn Pinv"}
METHOD_COLORS = {"contact_baseline": "#4C72B0",
                 "contact_hist8":    "#DD8452",
                 "dyn_pinv":         "#55A868"}
N_AGENTS = 5                       # agents per method
AGENT_CMAP = plt.cm.viridis        # per-agent shading in the stacked plots

# ---- outcome-replay constants (from real_robot_scripts/eval_config.yaml) ----
BREAK_FORCE      = 10.0            # N; force_mag >= this -> BREAK
XY_CENTERING     = 0.0025          # m; success xy tolerance
HOLE_HEIGHT      = 0.025           # m
SUCCESS_THRESH   = 0.2             # success if z_disp < HOLE_HEIGHT * SUCCESS_THRESH
Z_SUCCESS_LIMIT  = HOLE_HEIGHT * SUCCESS_THRESH     # = 0.005 m
TERMINATE_ON_SUCCESS = True

CI = 0.95                          # confidence level for the error bars

## Data loading & processing

`replay_episode` walks a per-step episode CSV exactly as `peg_insert_eval.run_episode`
does — registering the first terminal event and accumulating force metrics only while the
logical episode is live — so it works for the 14 agents that have only episode CSVs (no
`summary.*`). Output is one row per trajectory (`traj`, 300 rows) plus a per-agent
aggregate (`per_agent`, 15 rows).

In [ ]:
def replay_episode(df):
    "Replay run_episode's terminal + metric logic on a per-step episode DataFrame."
    succeeded = terminated = logical_done = False
    sum_force = max_force = 0.0
    data_length = 0
    for r in df.itertuples(index=False):
        if not logical_done:                        # metrics accrue through the logical end
            fm = float(r.force_mag)
            sum_force += fm
            max_force = max(max_force, fm)
            data_length += 1
        is_success = (r.xy_dist_to_target < XY_CENTERING) and (r.z_disp < Z_SUCCESS_LIMIT)
        if not logical_done:
            if not succeeded and is_success:        # success registered first (as in the eval)
                succeeded = True
                if TERMINATE_ON_SUCCESS:
                    logical_done = True
            if not logical_done and not terminated and fm >= BREAK_FORCE:
                terminated = True
                logical_done = True
        if logical_done:
            break
    outcome = "SUCCESS" if succeeded and not terminated else "BREAK" if terminated else "TIMEOUT"
    avg_force = sum_force / data_length if data_length else np.nan
    return dict(outcome=outcome, succeeded=succeeded, broke=terminated,
                avg_force=avg_force, max_force=max_force, data_length=data_length)


rows = []
for method in METHODS:
    for agent in range(N_AGENTS):
        eps = sorted(glob.glob(os.path.join(DATA_DIR, method, str(agent), "ep_*.csv")),
                     key=lambda p: int(os.path.basename(p)[3:-4]))
        for ep_path in eps:
            df = pd.read_csv(ep_path, usecols=["force_mag", "xy_dist_to_target", "z_disp"])
            m = replay_episode(df)
            m.update(method=method, agent=agent,
                     episode=int(os.path.basename(ep_path)[3:-4]))
            rows.append(m)

traj = pd.DataFrame(rows)
traj["timed_out"] = traj["outcome"] == "TIMEOUT"     # neither success nor break by max_steps
traj["method"] = pd.Categorical(traj["method"], categories=METHODS, ordered=True)
print(f"loaded {len(traj)} trajectories "
      f"({traj['method'].nunique()} methods x {N_AGENTS} agents x "
      f"{traj.groupby(['method','agent'], observed=True).size().iloc[0]} eps)")

# per-agent aggregate (the n=5 sample for the "among agents" plots)
per_agent = (traj.groupby(["method", "agent"], observed=True)
                 .agg(success_rate=("succeeded", "mean"),
                      break_rate=("broke", "mean"),
                      timeout_rate=("timed_out", "mean"),
                      avg_force=("avg_force", "mean"),   # mean per-traj mean force
                      mean_max_force=("max_force", "mean"),
                      peak_force=("max_force", "max"),   # agent's single worst peak
                      n_success=("succeeded", "sum"),
                      n_break=("broke", "sum"),
                      n_timeout=("timed_out", "sum"))
                 .reset_index())
per_agent["success_rate"] *= 100
per_agent["break_rate"]   *= 100
per_agent["timeout_rate"] *= 100
per_agent

## Plot 1 — averaged **among agents** (sample size = 5)

For each method the sample is the **5 per-agent values** (e.g. each agent's success rate over
its 20 trajectories). Bars are the across-agent mean; error bars are the 95% CI
(t-interval, n = 5). On the max-force panel a dot marks **each agent's peak force** (its single
largest `force_mag` across all 20 trajectories).

In [ ]:
def ci95_halfwidth(vals):
    "Half-width of the CI% confidence interval (t-based); 0 for n<2."
    v = np.asarray(vals, float)
    v = v[np.isfinite(v)]
    n = len(v)
    if n < 2:
        return 0.0
    sem = v.std(ddof=1) / np.sqrt(n)
    return float(stats.t.ppf(0.5 + CI / 2, n - 1) * sem)


def bar_ci(ax, samples_by_method, ylabel, title, dots_by_method=None, dots_label=None):
    "Bar of mean +/- CI% CI per method; optional per-agent dots overlaid."
    xs = np.arange(len(METHODS))
    means = [np.mean(samples_by_method[m]) for m in METHODS]
    errs  = [ci95_halfwidth(samples_by_method[m]) for m in METHODS]
    ax.bar(xs, means, yerr=errs, capsize=6,
           color=[METHOD_COLORS[m] for m in METHODS],
           edgecolor="black", linewidth=0.7, alpha=0.9,
           error_kw=dict(ecolor="black", lw=1.3))
    if dots_by_method is not None:
        for i, m in enumerate(METHODS):
            d = np.asarray(dots_by_method[m], float)
            jit = (np.arange(len(d)) - (len(d) - 1) / 2) * 0.05
            ax.scatter(np.full(len(d), xs[i]) + jit, d, s=32, color="black",
                       edgecolor="white", linewidth=0.6, zorder=5,
                       label=dots_label if i == 0 else None)
        if dots_label:
            ax.legend(fontsize=8, loc="upper right")
    ax.set_xticks(xs)
    ax.set_xticklabels([METHOD_LABELS[m] for m in METHODS])
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.margins(y=0.12)
    ax.grid(axis="y", alpha=0.3)


def method_samples(frame, col):
    "Dict {method: array of values} for a per-agent/per-trajectory column."
    return {m: frame.loc[frame["method"] == m, col].to_numpy() for m in METHODS}


# --- sample = 5 per-agent values ---
peaks = method_samples(per_agent, "peak_force")     # each agent's worst peak (5 dots)

fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
bar_ci(axes[0, 0], method_samples(per_agent, "success_rate"),
       "success rate (%)", "Average success rate")
bar_ci(axes[0, 1], method_samples(per_agent, "break_rate"),
       "break rate (%)", "Average break rate")
bar_ci(axes[1, 0], method_samples(per_agent, "avg_force"),
       "average force (N)", "Average force")
bar_ci(axes[1, 1], method_samples(per_agent, "mean_max_force"),
       "max force (N)", "Average max force",
       dots_by_method=peaks, dots_label="agent peak force")
axes[0, 0].set_ylim(0, 100)
axes[0, 1].set_ylim(0, 100)
fig.suptitle("Real-robot peg-insert — averaged among agents (n = 5, 95% CI)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "among_agents.svg"), bbox_inches="tight")
plt.show()

## Plot 2 — pooled **per trajectory** (sample size = 100)

Identical 2x2, but every trajectory is one sample: 100 per method (5 agents x 20).
Bars are the pooled mean; error bars are the 95% CI (t-interval, n = 100). The max-force
panel keeps the per-agent peak-force dots for reference.

In [ ]:
# per-trajectory samples: success/break as 0/1 over 100 trajectories
traj_pct = traj.assign(success_pct=traj["succeeded"] * 100.0,
                       break_pct=traj["broke"] * 100.0)

fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
bar_ci(axes[0, 0], method_samples(traj_pct, "success_pct"),
       "success rate (%)", "Average success rate")
bar_ci(axes[0, 1], method_samples(traj_pct, "break_pct"),
       "break rate (%)", "Average break rate")
bar_ci(axes[1, 0], method_samples(traj, "avg_force"),
       "average force (N)", "Average force")
bar_ci(axes[1, 1], method_samples(traj, "max_force"),
       "max force (N)", "Average max force",
       dots_by_method=peaks, dots_label="agent peak force")
axes[0, 0].set_ylim(0, 100)
axes[0, 1].set_ylim(0, 100)
fig.suptitle("Real-robot peg-insert — pooled per trajectory (n = 100, 95% CI)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "per_trajectory.svg"), bbox_inches="tight")
plt.show()

## Plot 3 — stacked success / break **counts**

One bar per method, stacked into 5 segments (one per agent), each segment the count of
successes (left) / breaks (right) for that agent out of its 20 trajectories. The count is
written on each segment.

In [ ]:
def stacked_counts(ax, count_col, title, ylabel):
    xs = np.arange(len(METHODS))
    colors = AGENT_CMAP(np.linspace(0.15, 0.9, N_AGENTS))
    bottoms = np.zeros(len(METHODS))
    for agent in range(N_AGENTS):
        vals = np.array([
            int(per_agent[(per_agent["method"] == m) & (per_agent["agent"] == agent)][count_col].iloc[0])
            for m in METHODS])
        ax.bar(xs, vals, bottom=bottoms, color=colors[agent],
               edgecolor="white", linewidth=0.8,
               label=f"agent {agent}")
        for x, v, b in zip(xs, vals, bottoms):
            if v > 0:
                ax.text(x, b + v / 2, str(int(v)), ha="center", va="center",
                        fontsize=9, fontweight="bold",
                        color="white" if agent < N_AGENTS - 2 else "black")
        bottoms += vals
    for x, tot in zip(xs, bottoms):                 # total on top of each stack
        ax.text(x, tot + 1, f"{int(tot)}", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_xticks(xs)
    ax.set_xticklabels([METHOD_LABELS[m] for m in METHODS])
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, N_AGENTS * 20 * 1.08)            # 5 agents x 20 eps
    ax.grid(axis="y", alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(12, 6))
stacked_counts(axes[0], "n_success", "Successes per agent", "number of successes")
stacked_counts(axes[1], "n_break", "Breaks per agent", "number of breaks")
axes[1].legend(fontsize=8, loc="upper right", title="agent")
fig.suptitle("Real-robot peg-insert — per-agent success / break counts (out of 20 each)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "stacked_counts.svg"), bbox_inches="tight")
plt.show()

## Plot 4 — timeouts

A trajectory **times out** when it neither succeeds nor breaks before `max_steps`
(`outcome == "TIMEOUT"`). Three views, mirroring the earlier plots:
- **left** — among agents (n = 5), mean timeout rate with 95% CI;
- **middle** — per trajectory (n = 100), pooled timeout rate with 95% CI;
- **right** — stacked per-agent timeout counts (one segment per agent).

> In this dataset every trajectory ends in SUCCESS or BREAK, so all timeout values are **0** —
> the panels are plotted on the same scales as the success/break plots to make that explicit.

In [ ]:
# per-trajectory timeout percent (0/1 -> %) for the middle (n=100) panel
traj_timeout = traj.assign(timeout_pct=traj["timed_out"] * 100.0)

fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.5))
bar_ci(axes[0], method_samples(per_agent, "timeout_rate"),
       "timeout rate (%)", "Among agents (n = 5, 95% CI)")
bar_ci(axes[1], method_samples(traj_timeout, "timeout_pct"),
       "timeout rate (%)", "Per trajectory (n = 100, 95% CI)")
axes[0].set_ylim(0, 100)
axes[1].set_ylim(0, 100)
stacked_counts(axes[2], "n_timeout", "Stacked timeout counts", "number of timeouts")
axes[2].legend(fontsize=8, loc="upper right", title="agent")
fig.suptitle("Real-robot peg-insert — timeouts", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "timeouts.svg"), bbox_inches="tight")
plt.show()